# Skyscanner URL Generator — Test Notebook

Bu notebook SerpAPI'den gelen uçuş verisini mevcut classları değiştirmeden
Skyscanner URL'sine dönüştürmeyi deniyor.

## Hedef URL formatları

**Genel arama (her zaman çalışır):**
```
https://www.skyscanner.com.tr/tasima/ucak-bileti/{dep}/{arr}/{YYMMDD}/?adultsv2=1&...
```

**Spesifik uçuş (config segmenti gerekiyor):**
```
https://www.skyscanner.com.tr/tasima/ucak-bileti/{dep}/{arr}/{YYMMDD}/config/{carrier_id}-{YYMMDDHHMM}--{dep_airport_id}-{stops}-{arr_airport_id}-{YYMMDDHHMM}?...
```

Config segmentindeki carrier ve airport ID'leri Skyscanner'ın kendi iç numerik kodları.
Bu notebook her iki yaklaşımı da test eder.

In [13]:
import os
import serpapi
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("SERPAPI_KEY")

## 1. Skyscanner URL üretici fonksiyonlar

In [14]:
SKYSCANNER_BASE = "https://www.skyscanner.com.tr/tasima/ucak-bileti"
SKYSCANNER_PARAMS = (
    "adultsv2=1&cabinclass=economy&childrenv2=&ref=home"
    "&rtn=0&outboundaltsenabled=false&inboundaltsenabled=false"
    "&currency=EUR&preferdirects=false"
)


def _date_to_yymmdd(date_str: str) -> str:
    """'2026-05-05' veya '2026-05-05 09:10' -> '260505'"""
    d = date_str[:10]
    return d[2:].replace("-", "")


def _datetime_to_yymmddhhmm(dt_str: str) -> str:
    """'2026-05-05 09:10' -> '2605050910'"""
    date_part, time_part = dt_str.strip().split()
    yy = date_part[2:4]
    mm = date_part[5:7]
    dd = date_part[8:10]
    hh = time_part[:2]
    mi = time_part[3:5]
    return f"{yy}{mm}{dd}{hh}{mi}"


def skyscanner_general_url(
    departure_iata: str,
    arrival_iata: str,
    date_str: str,
    currency: str = "EUR",
) -> str:
    """
    Her zaman çalışan genel arama URL'si.
    dep/arr → nihai kalkış ve varış IATA kodu (aktarmalıda da final destinasyon).
    """
    dep = departure_iata.lower()
    arr = arrival_iata.lower()
    date_compact = _date_to_yymmdd(date_str)
    params = SKYSCANNER_PARAMS.replace("currency=EUR", f"currency={currency}")
    return f"{SKYSCANNER_BASE}/{dep}/{arr}/{date_compact}/?{params}"


def skyscanner_config_url(
    departure_iata: str,
    arrival_iata: str,      # nihai varış (aktarmalıda son bacak)
    departure_dt: str,      # ilk bacak kalkış: 'YYYY-MM-DD HH:MM'
    arrival_dt: str,        # son bacak varış: 'YYYY-MM-DD HH:MM'
    flight_number: str,     # ilk bacak uçuş numarası: 'LH 1132'
    stops: int = 0,
    currency: str = "EUR",
) -> str:
    """
    Spesifik uçuşa yönlendiren /config/ URL denemesi.

    NOT: Config segmentindeki sayısal ID'ler (carrier, airport)
    Skyscanner'ın kendi iç veritabanından geliyor. Burada flight_number +
    IATA kodları kullanılır — Skyscanner doğrulayamazsa genel arama sayfasına düşer.
    """
    dep = departure_iata.lower()
    arr = arrival_iata.lower()            # nihai destinasyon
    date_compact = _date_to_yymmdd(departure_dt)
    dep_dt = _datetime_to_yymmddhhmm(departure_dt)
    arr_dt = _datetime_to_yymmddhhmm(arrival_dt)  # son bacak varışı

    flight_num = flight_number.replace(" ", "").upper()  # 'LH1132'

    # Format: {flight_num}-{dep_yymmddhhmm}--{dep_iata}-{stops}-{arr_iata}-{arr_yymmddhhmm}
    config_segment = f"{flight_num}-{dep_dt}--{dep.upper()}-{stops}-{arr.upper()}-{arr_dt}"
    params = SKYSCANNER_PARAMS.replace("currency=EUR", f"currency={currency}")
    return f"{SKYSCANNER_BASE}/{dep}/{arr}/{date_compact}/config/{config_segment}?{params}"


def build_skyscanner_url(flight_dict: dict, prefer_config: bool = True, currency: str = "EUR") -> str:
    """
    SerpAPI raw flight dict'inden Skyscanner URL üretir.
    Aktarmalı uçuşlarda: ilk bacağın kalkışı + son bacağın varışı kullanılır.
    URL path'indeki arrival da nihai destinasyondur (ara durak değil).
    """
    first_leg = flight_dict["flights"][0]
    last_leg  = flight_dict["flights"][-1]   # direkt uçuşta aynı, aktarmalıda son bacak

    dep_iata = first_leg["departure_airport"]["id"]
    arr_iata = last_leg["arrival_airport"]["id"]     # nihai destinasyon (ör: NUE, AMS değil)
    dep_dt   = first_leg["departure_airport"]["time"]
    arr_dt   = last_leg["arrival_airport"]["time"]   # nihai varış saati
    flight_number = first_leg.get("flight_number", "")
    stops    = len(flight_dict["flights"]) - 1

    if prefer_config and flight_number:
        return skyscanner_config_url(dep_iata, arr_iata, dep_dt, arr_dt, flight_number, stops, currency)
    return skyscanner_general_url(dep_iata, arr_iata, dep_dt, currency)


print("Fonksiyonlar tanımlandı ✓")


Fonksiyonlar tanımlandı ✓


## 2. Hızlı URL formatı testi (API çağrısı olmadan)

## 2. Skyscanner Entity ID Sorunu

Config URL'nin çalışması için IATA kodu değil, **Skyscanner'ın kendi sayısal ID'leri** gerekiyor:

| Alan | IATA (işe yaramaz) | Skyscanner Entity ID (gerekli) |
|---|---|---|
| Carrier | `KL` / `KL1628` | `17468` |
| Dep airport | `VCE` | `32132` |
| Arr airport | `NUE` | `14827` |

Bu ID'ler **Skyscanner URL'lerinden gözlemlenerek** toplanabiliyor.
Aşağıdaki cell'de bilinen ID'ler bir lookup table'a yazılıyor; bilinmeyenler `skyscanner_general_url`'e fallback yapıyor.


In [15]:
# ── Skyscanner Entity ID Lookup ───────────────────────────────────────────────
# Config URL çalışması için gereken sayısal ID'ler.
# Yeni bir ID öğrendiğinde buraya ekle: Skyscanner URL'sinden kopyala.
# Format: config/{carrier_id}-{YYMMDDHHMM}--{dep_airport_id}-{stops}-{arr_airport_id}-{YYMMDDHHMM}

AIRPORT_ENTITY_IDS: dict[str, str] = {
    # Kullanıcı örneğinden doğrulanmış
    "VCE": "32132",   # Venice Marco Polo
    "NUE": "14827",   # Nuremberg
    # Buraya yenilerini ekle → Skyscanner'da bir uçuş ara, URL'den al
}

CARRIER_ENTITY_IDS: dict[str, str] = {
    # Havayolu IATA kodu (2 harf) → Skyscanner carrier entity ID
    "KL": "17468",    # KLM  (kullanıcı örneğinden doğrulanmış)
    # Buraya yenilerini ekle
}


def _parse_carrier_iata(flight_number: str) -> str:
    """'KL 1628' veya 'LH1132' → 'KL' """
    parts = flight_number.strip().split()
    if parts:
        # "LH1132" gibi birleşik yazılmışsa ilk 2 harfi al
        return parts[0][:2].upper()
    return ""


def build_skyscanner_url_v2(flight_dict: dict, currency: str = "EUR") -> tuple[str, str]:
    """
    Entity ID tablosunu kullanarak config URL üretir.
    Eksik ID varsa general URL'e fallback yapar.

    Returns:
        (url, url_type)  — url_type: 'config' | 'general'
    """
    first_leg = flight_dict["flights"][0]
    last_leg  = flight_dict["flights"][-1]

    dep_iata = first_leg["departure_airport"]["id"]
    arr_iata = last_leg["arrival_airport"]["id"]
    dep_dt   = first_leg["departure_airport"]["time"]
    arr_dt   = last_leg["arrival_airport"]["time"]
    flight_number = first_leg.get("flight_number", "")
    stops    = len(flight_dict["flights"]) - 1

    carrier_iata   = _parse_carrier_iata(flight_number)
    dep_entity     = AIRPORT_ENTITY_IDS.get(dep_iata)
    arr_entity     = AIRPORT_ENTITY_IDS.get(arr_iata)
    carrier_entity = CARRIER_ENTITY_IDS.get(carrier_iata)

    if dep_entity and arr_entity and carrier_entity:
        dep_dt_c = _datetime_to_yymmddhhmm(dep_dt)
        arr_dt_c = _datetime_to_yymmddhhmm(arr_dt)
        date_c   = _date_to_yymmdd(dep_dt)
        config   = f"{carrier_entity}-{dep_dt_c}--{dep_entity}-{stops}-{arr_entity}-{arr_dt_c}"
        params   = SKYSCANNER_PARAMS.replace("currency=EUR", f"currency={currency}")
        url      = f"{SKYSCANNER_BASE}/{dep_iata.lower()}/{arr_iata.lower()}/{date_c}/config/{config}?{params}"
        return url, "config ✅"
    else:
        missing = []
        if not dep_entity:     missing.append(f"airport:{dep_iata}")
        if not arr_entity:     missing.append(f"airport:{arr_iata}")
        if not carrier_entity: missing.append(f"carrier:{carrier_iata}")
        url = skyscanner_general_url(dep_iata, arr_iata, dep_dt, currency)
        return url, f"general ⚠️ (eksik entity ID → {', '.join(missing)})"


# ── Hızlı doğrulama: kullanıcının orijinal URL'siyle karşılaştır ──────────────
test_flight = {
    "flights": [
        {
            "departure_airport": {"id": "VCE", "time": "2026-05-10 06:00"},
            "arrival_airport":   {"id": "AMS", "time": "2026-05-10 07:50"},
            "flight_number": "KL 1628",
        },
        {
            "departure_airport": {"id": "AMS", "time": "2026-05-10 12:25"},
            "arrival_airport":   {"id": "NUE", "time": "2026-05-10 13:40"},
            "flight_number": "KL 1843",
        },
    ],
}

url, url_type = build_skyscanner_url_v2(test_flight)
print(f"Tür   : {url_type}")
print(f"URL   : {url}")
print()
print("Orijinal:")
print("https://www.skyscanner.com.tr/tasima/ucak-bileti/vce/nue/260510/config/17468-2605100600--32132-1-14827-2605101340?adultsv2=1&cabinclass=economy&childrenv2=&ref=home&rtn=0&outboundaltsenabled=false&inboundaltsenabled=false&preferdirects=false")


Tür   : config ✅
URL   : https://www.skyscanner.com.tr/tasima/ucak-bileti/vce/nue/260510/config/17468-2605100600--32132-1-14827-2605101340?adultsv2=1&cabinclass=economy&childrenv2=&ref=home&rtn=0&outboundaltsenabled=false&inboundaltsenabled=false&currency=EUR&preferdirects=false

Orijinal:
https://www.skyscanner.com.tr/tasima/ucak-bileti/vce/nue/260510/config/17468-2605100600--32132-1-14827-2605101340?adultsv2=1&cabinclass=economy&childrenv2=&ref=home&rtn=0&outboundaltsenabled=false&inboundaltsenabled=false&preferdirects=false


In [10]:
# Kullanıcının verdiği örneği doğrula
general = skyscanner_general_url("VCE", "NUE", "2026-05-05")
print("Genel URL:")
print(general)
print()

config = skyscanner_config_url(
    departure_iata="VCE",
    arrival_iata="NUE",
    departure_dt="2026-05-05 09:10",
    arrival_dt="2026-05-05 19:40",
    flight_number="LH 1132",
    stops=0,
)
print("Config URL (IATA tabanlı deneme):")
print(config)

Genel URL:
https://www.skyscanner.com.tr/tasima/ucak-bileti/vce/nue/260505/?adultsv2=1&cabinclass=economy&childrenv2=&ref=home&rtn=0&outboundaltsenabled=false&inboundaltsenabled=false&currency=EUR&preferdirects=false

Config URL (IATA tabanlı deneme):
https://www.skyscanner.com.tr/tasima/ucak-bileti/vce/nue/260505/config/LH1132-2605050910--VCE-0-NUE-2605051940?adultsv2=1&cabinclass=economy&childrenv2=&ref=home&rtn=0&outboundaltsenabled=false&inboundaltsenabled=false&currency=EUR&preferdirects=false


## 3. Gerçek uçuş araması + Skyscanner URL ekleme

`flight_search` ve `find_cheap_flight_plus_ground` classlarına dokunulmadan
raw SerpAPI verisi üzerinde deniyoruz.

In [16]:
# Parametreler — buradan değiştir
DEPARTURE = "VCE"      # Venice Marco Polo
ARRIVAL   = "NUE"      # Nuremberg
DATE      = "2026-05-10"

client = serpapi.Client(api_key=API_KEY)
results = client.search({
    "engine": "google_flights",
    "departure_id": DEPARTURE,
    "arrival_id": ARRIVAL,
    "currency": "EUR",
    "type": "2",
    "outbound_date": DATE,
    "no_cache": False,
})

best_flights  = results.get("best_flights", [])
other_flights = results.get("other_flights", [])
all_flights   = [(f, "Best") for f in best_flights] + [(f, "Other") for f in other_flights]

print(f"Toplam uçuş: {len(all_flights)} ({len(best_flights)} best, {len(other_flights)} other)")

Toplam uçuş: 18 (3 best, 15 other)


In [17]:
rows = []
for flight, ftype in all_flights:
    first_leg = flight["flights"][0]
    last_leg  = flight["flights"][-1]

    url, url_type = build_skyscanner_url_v2(flight)

    rows.append({
        "type":           ftype,
        "airline":        first_leg.get("airline"),
        "flight_number":  first_leg.get("flight_number", ""),
        "price":          flight.get("price"),
        "departure_time": first_leg["departure_airport"]["time"],
        "arrival_time":   last_leg["arrival_airport"]["time"],
        "stops":          len(flight["flights"]) - 1,
        "duration_min":   flight.get("total_duration"),
        "url_type":       url_type,
        "skyscanner_url": url,
    })

df = pd.DataFrame(rows).sort_values("price").reset_index(drop=True)
df[["type", "airline", "flight_number", "price", "departure_time", "arrival_time", "stops", "url_type"]]


,type,airline,flight_number,price,departure_time,arrival_time,stops,url_type
0,Other,Air France,AF 1327,365,2026-05-10 05:55,2026-05-10 22:15,1,general ⚠️ (eksik entity ID → carrier:AF)
1,Other,Air France,AF 1327,369,2026-05-10 05:55,2026-05-10 17:55,1,general ⚠️ (eksik entity ID → carrier:AF)
2,Best,KLM,KL 1628,398,2026-05-10 06:00,2026-05-10 13:40,1,config ✅
3,Other,Air Serbia,JU 465,465,2026-05-10 20:30,2026-05-11 20:05,1,general ⚠️ (eksik entity ID → carrier:JU)
4,Best,Air France,AF 1327,514,2026-05-10 05:55,2026-05-10 10:20,1,general ⚠️ (eksik entity ID → carrier:AF)
5,Other,Lufthansa,LH 333,530,2026-05-10 06:30,2026-05-10 17:15,2,general ⚠️ (eksik entity ID → carrier:LH)
6,Other,Lufthansa,LH 333,530,2026-05-10 06:30,2026-05-10 17:15,2,general ⚠️ (eksik entity ID → carrier:LH)
7,Other,Lufthansa,LH 333,530,2026-05-10 06:30,2026-05-10 17:15,2,general ⚠️ (eksik entity ID → carrier:LH)
8,Other,Air Dolomiti,EN 8205,550,2026-05-10 17:15,2026-05-10 21:30,1,general ⚠️ (eksik entity ID → carrier:EN)
9,Other,Air Dolomiti,EN 8201,550,2026-05-10 10:00,2026-05-10 17:15,1,general ⚠️ (eksik entity ID → carrier:EN)


## 4. URL'leri tıklanabilir olarak görüntüle

In [18]:
from IPython.display import display, HTML

rows_html = []
for _, row in df.iterrows():
    bg = "#e8f5e9" if "config" in row["url_type"] else "#fff8e1"
    rows_html.append(
        f"<tr style='background:{bg}'>"
        f"<td>{row['type']}</td>"
        f"<td>{row['airline']}</td>"
        f"<td>{row['flight_number']}</td>"
        f"<td><b>{row['price']} €</b></td>"
        f"<td>{row['departure_time']}</td>"
        f"<td>{row['arrival_time']}</td>"
        f"<td>{row['stops']}</td>"
        f"<td>{row['url_type']}</td>"
        f"<td><a href='{row['skyscanner_url']}' target='_blank'>Aç</a></td>"
        f"</tr>"
    )

table = (
    "<table border='1' cellpadding='6'>"
    "<tr><th>Type</th><th>Airline</th><th>Flight#</th><th>Price</th>"
    "<th>Departure</th><th>Arrival</th><th>Stops</th><th>URL Türü</th><th>Link</th></tr>"
    + "".join(rows_html)
    + "</table>"
    "<br><small>🟢 config ✅ = entity ID biliniyor, spesifik uçuş linki &nbsp; "
    "🟡 general ⚠️ = entity ID eksik, genel arama sayfasına gider</small>"
)
display(HTML(table))


Type,Airline,Flight#,Price,Departure,Arrival,Stops,URL Türü,Link
Other,Air France,AF 1327,365 €,2026-05-10 05:55,2026-05-10 22:15,1,general ⚠️ (eksik entity ID → carrier:AF),Aç
Other,Air France,AF 1327,369 €,2026-05-10 05:55,2026-05-10 17:55,1,general ⚠️ (eksik entity ID → carrier:AF),Aç
Best,KLM,KL 1628,398 €,2026-05-10 06:00,2026-05-10 13:40,1,config ✅,Aç
Other,Air Serbia,JU 465,465 €,2026-05-10 20:30,2026-05-11 20:05,1,general ⚠️ (eksik entity ID → carrier:JU),Aç
Best,Air France,AF 1327,514 €,2026-05-10 05:55,2026-05-10 10:20,1,general ⚠️ (eksik entity ID → carrier:AF),Aç
Other,Lufthansa,LH 333,530 €,2026-05-10 06:30,2026-05-10 17:15,2,general ⚠️ (eksik entity ID → carrier:LH),Aç
Other,Lufthansa,LH 333,530 €,2026-05-10 06:30,2026-05-10 17:15,2,general ⚠️ (eksik entity ID → carrier:LH),Aç
Other,Lufthansa,LH 333,530 €,2026-05-10 06:30,2026-05-10 17:15,2,general ⚠️ (eksik entity ID → carrier:LH),Aç
Other,Air Dolomiti,EN 8205,550 €,2026-05-10 17:15,2026-05-10 21:30,1,general ⚠️ (eksik entity ID → carrier:EN),Aç
Other,Air Dolomiti,EN 8201,550 €,2026-05-10 10:00,2026-05-10 17:15,1,general ⚠️ (eksik entity ID → carrier:EN),Aç


## 5. explore_search (flight_and_ground için) ile test

Yakındaki ucuz uçuşlar için de Skyscanner URL'i üret.

In [19]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lam = math.radians(lon2 - lon1)
    a = math.sin(d_phi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(d_lam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


explore_results = client.search({
    "engine": "google_travel_explore",
    "departure_id": DEPARTURE,
    "currency": "EUR",
    "type": "2",
    "outbound_date": DATE,
    "no_cache": False,
})

# Hedef: Nuremberg (lat/lon)
TARGET_LAT, TARGET_LON = 49.4521, 11.0767
MAX_KM = 400

rows_explore = []
for dest in explore_results.get("destinations", []):
    if "flight_price" not in dest:
        continue
    lat = dest["gps_coordinates"]["latitude"]
    lon = dest["gps_coordinates"]["longitude"]
    dist = haversine_km(lat, lon, TARGET_LAT, TARGET_LON)
    if dist > MAX_KM:
        continue
    iata = dest.get("destination_airport", {}).get("code", "")
    rows_explore.append({
        "city":        dest["name"],
        "iata":        iata,
        "price":       dest["flight_price"],
        "duration":    dest["flight_duration"],
        "airline":     dest["airline"],
        "distance_km": round(dist, 1),
        "skyscanner_url": skyscanner_general_url(DEPARTURE, iata, DATE) if iata else "",
    })

df_explore = pd.DataFrame(rows_explore).sort_values("price").reset_index(drop=True)
df_explore

HTTPError: 503 Server Error: Service Unavailable for url: https://serpapi.com/search?engine=google_travel_explore&departure_id=VCE&currency=EUR&type=2&outbound_date=2026-05-10&no_cache=False&api_key=6e737ea761d241b180cba3b389d77cf0d0acb9001e4888cea11c967e057585ab

In [ ]:
rows_html2 = []
for _, row in df_explore.iterrows():
    rows_html2.append(
        f"<tr>"
        f"<td>{row['city']} ({row['iata']})</td>"
        f"<td><b>{row['price']} €</b></td>"
        f"<td>{row['airline']}</td>"
        f"<td>{row['distance_km']} km</td>"
        f"<td><a href='{row['skyscanner_url']}' target='_blank'>Skyscanner</a></td>"
        f"</tr>"
    )

table2 = (
    "<table border='1' cellpadding='6'>"
    "<tr><th>City</th><th>Price</th><th>Airline</th><th>Distance</th><th>URL</th></tr>"
    + "".join(rows_html2)
    + "</table>"
)
display(HTML(table2))

## Sonuç & Notlar

| URL Türü | Durum | Açıklama |
|---|---|---|
| `skyscanner_general_url` | ✅ Çalışır | IATA + tarih yeterli, Skyscanner arama sayfasına gider |
| `skyscanner_config_url` | ⚠️ Deneysel | Config segmentindeki sayısal ID'ler Skyscanner'ın iç kodları — IATA tabanlı yaklaşım eşleşmeyebilir |

### Config URL için tam çözüm seçenekleri
1. **Skyscanner widget/embed API** — oturum açmadan config ID'lerini almak için kullanılabilir (rate-limited)
2. **Skyscanner Affiliates API** — partner hesabıyla tam leg ID'lerine erişim
3. **Şu an en pragmatik**: `skyscanner_general_url` her uçuş için iş görür, kullanıcıyı doğru güne ve rotaya yönlendirir


## Entity ID Tablosunu Genişletme

Yeni bir havayolu veya havalimanı için entity ID'yi bulmak çok basit:

1. [skyscanner.com.tr](https://www.skyscanner.com.tr) üzerinde o rotayı ara
2. İstediğin spesifik uçuşa tıkla → URL şu formata geçer:
   ```
   /config/{carrier_id}-{YYMMDDHHMM}--{dep_airport_id}-{stops}-{arr_airport_id}-{YYMMDDHHMM}
   ```
3. Sayıları kopyalayıp aşağıdaki cell'e ekle


In [ ]:
# ── Yeni Entity ID Ekle ────────────────────────────────────────────────────────
# Skyscanner'dan gözlemlediğin yeni ID'leri buraya ekle, yukarıdaki tablolara işlenecek.

def register_airport(iata: str, entity_id: str):
    AIRPORT_ENTITY_IDS[iata.upper()] = entity_id
    print(f"Eklendi: {iata.upper()} → {entity_id}")

def register_carrier(carrier_iata: str, entity_id: str):
    CARRIER_ENTITY_IDS[carrier_iata.upper()] = entity_id
    print(f"Eklendi: {carrier_iata.upper()} → {entity_id}")


# Örnek kullanım:
# register_airport("FRA", "13554")
# register_airport("MUC", "14231")
# register_carrier("LH", "1121")

print("Mevcut havalimanı ID'leri:", AIRPORT_ENTITY_IDS)
print("Mevcut carrier ID'leri   :", CARRIER_ENTITY_IDS)
